# V1 · 基础时序版：先让链路跑通

这是整个项目的基线版本，目标只有一个：**把“文本 → LLM → JSON → Neo4j → Cypher”这条链路完整跑通**。

这一版只做三件事：

1. 从文本中抽取 `event` / `time` / `location` / `participants`；
2. 建立 `Event` 与 `Person` 节点；
3. 按原文顺序连接 `FOLLOWED_BY`，人物连接 `PARTICIPATED_IN`。

> 这一版不写因果。它的价值是“通”，不是“对”。

**运行前检查**

- 已复制 `.env.example` 为 `.env`，并填好 `DEEPSEEK_API_KEY` / `NEO4J_PASSWORD`；
- Neo4j 已启动，`bolt://localhost:7687` 可连接；
- 当前 Notebook 从仓库根目录或 `notebooks/` 启动都可以。

In [ ]:
# ============================================================
# 0. 环境与路径：所有密钥都从 .env 读取，Notebook 里不写死
# ============================================================
import os
from pathlib import Path

from dotenv import load_dotenv

# Notebook 可能从 notebooks/ 启动，也可能从仓库根目录启动
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

load_dotenv(PROJECT_ROOT / ".env")

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY", "").strip()
DEEPSEEK_MODEL = os.getenv("DEEPSEEK_MODEL", "deepseek-chat")
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "")

# 默认不删库。第一次跑、或想从零复现时，才在 .env 里设 RESET_DB=true
RESET_DB = os.getenv("RESET_DB", "false").lower() == "true"

if not DEEPSEEK_API_KEY:
    raise RuntimeError("缺少 DEEPSEEK_API_KEY：请复制 .env.example 为 .env 并填写。")
if not NEO4J_PASSWORD:
    raise RuntimeError("缺少 NEO4J_PASSWORD：请复制 .env.example 为 .env 并填写。")

DATA_PATH = PROJECT_ROOT / "data" / "xuanwumen_text.txt"
if not DATA_PATH.exists():
    raise FileNotFoundError(f"找不到文本文件：{DATA_PATH}")

print("仓库根目录:", PROJECT_ROOT)
print("文本文件:", DATA_PATH)
print("Neo4j:", NEO4J_URI, "| 用户:", NEO4J_USER)
print("RESET_DB:", RESET_DB)


## 1. 读入玄武门文本

文本放在 `data/xuanwumen_text.txt`，是一段约 600 字的教学用历史摘要，不是史料原文。

In [ ]:
text = DATA_PATH.read_text(encoding="utf-8").strip()
print(f"文本长度：{len(text)} 字符")
print("-" * 60)
print(text[:300] + ("..." if len(text) > 300 else ""))


In [ ]:
# ============================================================
# 2. 初始化 DeepSeek
# ============================================================
# 用官方 langchain-deepseek，而不是把 OpenAI 客户端硬指到 DeepSeek：
# 前者的字段、重试和流式行为与 DeepSeek API 更匹配。
from langchain_deepseek import ChatDeepSeek

llm = ChatDeepSeek(
    model=DEEPSEEK_MODEL,
    api_key=DEEPSEEK_API_KEY,
    temperature=0,  # 抽取任务尽量确定性
)
print("LLM 初始化完成:", DEEPSEEK_MODEL)


## 2. 只抽取“事件 + 时间 + 地点 + 参与者”

Prompt 里明确写了“不要补充常识或推断”。这不是装饰：V1 的目标是拿到**可回溯到原文**的事实字段。

In [ ]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    """你是一个严谨的历史事件抽取助手。请从下面的文本中抽取事件，只输出一个 JSON 对象，不要输出 Markdown 代码块或解释。

JSON 格式：
{{"events": [
  {{"event": "主谓短语，不超过 30 字", "time": "原文中的时间；没有则 null", "location": "原文中的地点；没有则 null", "participants": ["原文中出现的人名"]}}
]}}

抽取规则：
1. 按原文叙事顺序输出；
2. 只抽取原文明确写到的事件，不要补充常识或推断；
3. participants 只填原文出现的人名，不要填头衔；
4. 时间、地点无法确定时填 null，不要猜；
5. 事件数量控制在 10~20 个之间。

文本：
{text}"""
)
print("Prompt 已就绪")


In [ ]:
# ============================================================
# 3. 调用模型并做最小规则清洗
# ============================================================
import json

from langchain_core.output_parsers import JsonOutputParser

chain = prompt | llm | JsonOutputParser()

try:
    raw = chain.invoke({"text": text})
except Exception as exc:
    raise RuntimeError(
        "模型没有返回合法 JSON。请检查 API Key、模型名，"
        "或把 temperature 设为 0 后重试。"
    ) from exc

raw_events = raw.get("events", []) if isinstance(raw, dict) else []
if not isinstance(raw_events, list) or not raw_events:
    raise RuntimeError(f"抽取结果为空或格式异常：{raw!r}")


def normalize_events(items):
    """把模型输出清洗成稳定的内部结构。"""
    cleaned = []
    for item in items:
        if not isinstance(item, dict):
            continue
        name = str(item.get("event") or "").strip()
        if not name:
            continue

        participants = item.get("participants") or []
        if not isinstance(participants, list):
            participants = [participants]
        participants = [str(p).strip() for p in participants if str(p).strip()]

        cleaned.append({
            "event": name,
            "time": item.get("time") or None,
            "location": item.get("location") or None,
            "participants": sorted(set(participants)),
        })
    return cleaned


events = normalize_events(raw_events)
if not events:
    raise RuntimeError("清洗后没有有效事件。")

print(f"抽取到 {len(events)} 个事件")
for ev in events[:3]:
    print(json.dumps(ev, ensure_ascii=False))


In [ ]:
# ============================================================
# 3. 连接 Neo4j
# ============================================================
from neo4j import GraphDatabase

driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USER, NEO4J_PASSWORD),
)
driver.verify_connectivity()
print("Neo4j 连接成功")


## 3. 建图：Event / Person / FOLLOWED_BY

`MERGE` 保证重复运行不会重复建点；`RESET_DB=true` 时才会清空数据库。

事件 ID 用 `xuanwu_v1_00` 这种**由代码生成**的稳定 ID，不依赖模型返回的字符串。

In [ ]:
# ============================================================
# 4. 约束、清库与建图
# ============================================================
def ensure_constraints(tx):
    tx.run("""
        CREATE CONSTRAINT event_id_unique IF NOT EXISTS
        FOR (e:Event) REQUIRE e.id IS UNIQUE
    """)
    tx.run("""
        CREATE CONSTRAINT person_name_unique IF NOT EXISTS
        FOR (p:Person) REQUIRE p.name IS UNIQUE
    """)


def reset_database(tx):
    tx.run("MATCH (n) DETACH DELETE n")


def build_v1(tx, ev_list):
    for order, ev in enumerate(ev_list):
        event_id = f"xuanwu_v1_{order:02d}"

        # 1) 事件节点
        tx.run("""
            MERGE (e:Event {id: $id})
            SET e.name = $name,
                e.time = $time,
                e.location = $location,
                e.order = $order,
                e.source = 'llm_extracted'
        """, id=event_id, name=ev["event"], time=ev["time"],
             location=ev["location"], order=order)

        # 2) 人物节点 + 参与关系
        for person in ev["participants"]:
            tx.run("""
                MERGE (p:Person {name: $name})
                WITH p
                MATCH (e:Event {id: $event_id})
                MERGE (p)-[r:PARTICIPATED_IN]->(e)
                SET r.source = 'llm_extracted'
            """, name=person, event_id=event_id)

        # 3) 原文顺序的时序线：前一事件 -> 当前事件
        if order > 0:
            prev_id = f"xuanwu_v1_{order - 1:02d}"
            tx.run("""
                MATCH (prev:Event {id: $prev_id})
                MATCH (curr:Event {id: $curr_id})
                MERGE (prev)-[r:FOLLOWED_BY]->(curr)
                SET r.source = 'text_order'
            """, prev_id=prev_id, curr_id=event_id)


with driver.session() as session:
    session.execute_write(ensure_constraints)
    if RESET_DB:
        session.execute_write(reset_database)
        print("已清空数据库（RESET_DB=true）")
    session.execute_write(build_v1, events)

print("V1 建图完成")


## 4. 验收：点、边、时间线

注意：这里的 `MATCH p=(:Event)-[:FOLLOWED_BY*]->(:Event)` 只取一条最长路径，用来肉眼确认顺序；
生产环境不要写无上限的可变长路径。

In [ ]:
with driver.session() as session:
    node_count = session.run("MATCH (n) RETURN count(n) AS c").single()["c"]
    rel_count = session.run("MATCH ()-[r]->() RETURN count(r) AS c").single()["c"]
    path_record = session.run("""
        MATCH p=(:Event)-[:FOLLOWED_BY*]->(:Event)
        RETURN p
        LIMIT 1
    """).single()

print(f"节点数：{node_count}")
print(f"关系数：{rel_count}")
if path_record:
    names = [n["name"] for n in path_record["p"].nodes]
    print("时间线样例：", " -> ".join(names))
else:
    print("时间线样例：无（可能没有连成链）")


In [ ]:
# ============================================================
# 5. 在 Neo4j Browser 里执行这些查询
# ============================================================
queries = [
    "MATCH p=()-[r]->() RETURN p LIMIT 100;",
    "MATCH p=(:Event)-[:FOLLOWED_BY*]->(:Event) RETURN p LIMIT 1;",
    "MATCH (e:Event) RETURN e.name, e.time ORDER BY e.order;",
]
for q in queries:
    print(q)


## 这一版得到了什么

- 一条可查询的事件时间线；
- 人物挂在事件上，能回答“谁参与了什么”；
- 一套稳定的 ID / 约束 / 幂等写入方式，为 V2、V3 复用。

## 这一版缺什么

- 没有因果：只能看到“先后”，看不到“为什么”；
- 没有人物之间的关系：李世民和李建成都挂在事件上，但图里没有“政敌”这条边；
- 单线程时间线：所有事件被强行串成一条链，原文若有多条并行叙事，会被压平。

下一步进入 V2：让模型输出 `cause`，然后用 `SequenceMatcher` 把原因对齐到事件。
**那一版会翻车，而且翻得很有代表性。**

In [ ]:
driver.close()
print("Neo4j 连接已关闭")
